# Behavior-to-Grid Flexibility Certificate (BGFC)

## Reproducible synthetic demonstration

This notebook demonstrates how an uncertain behavioral response can be converted into an auditable flexibility certificate for workplace electric-vehicle charging. The certificate reports how much flexible energy may be committed, how often that commitment is delivered, how severe delivery shortfalls are, and when the controller abstains because conditions are too uncertain.

The workflow combines:

1. a behaviorally bounded response correspondence;
2. a linearized IEEE 33-bus radial-feeder model;
3. synthetic load, forecast-error, and response scenarios;
4. static and adaptive certificate policies;
5. held-out reliability calibration;
6. in-distribution and out-of-distribution auditing; and
7. paired statistical comparisons on identical scenarios.

### Evidence boundary

The bunching estimate $b=1.84$ with 95% interval $[0.99,2.70]$ is an imported numerical input. All feeder profiles, response draws, forecast errors, service values, and controller comparisons generated here are synthetic. The notebook is a computational demonstration and should not be interpreted as field validation, causal identification, or full AC-network certification.


## Quick start

Run the notebook from top to bottom. A fixed random seed makes the synthetic experiment reproducible.

### Supported environments

- **Google Colab:** Google Drive is mounted automatically, and outputs are written to `MyDrive/Outputs/BGFC/<timestamp>/`.
- **Local Python:** if Colab is unavailable, outputs are written beneath the current working directory in `Outputs/BGFC/<timestamp>/`.

### Python dependencies

The notebook uses only common scientific Python packages: `numpy`, `pandas`, and `matplotlib`. It also uses Python's standard-library modules for paths, dates, JSON, statistics, and archive creation.

### Output structure

Each run creates raw scenario results, audit tables, figures in PDF and PNG formats, machine-readable metadata, LaTeX table exports, a technical summary, and a compressed results archive. The console output always prints the exact run directory.

### Navigation

1. Environment and output directories
2. Feeder and baseline-controller experiment
3. Certificate metrics and static audit
4. Adaptive held-out certificate
5. Common-population inference and exports
6. Interpretation and extension guidance


In [ ]:
# Google Colab / Google Drive setup
from pathlib import Path
from datetime import datetime
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT=Path('/content/drive/MyDrive')
except ImportError:
    DRIVE_ROOT=Path.cwd()
PROJECT_NAME='BGFC'
RUN_ID=datetime.now().strftime('%Y_%m_%d_%H_%M_%S')
RUN_DIR=DRIVE_ROOT/'Outputs'/PROJECT_NAME/RUN_ID
RUN_DIR.mkdir(parents=True,exist_ok=True)
print('Output directory:',RUN_DIR)


## 1. Feeder and baseline-controller experiment

This section defines the synthetic IEEE 33-bus feeder, a 24-hour base-load profile, workplace EV charging at three buses, and an energy-neutral charging-shift action. It then generates 400 scenarios with behavioral-response and forecast uncertainty.

Five initial policies are evaluated:

- **Uncontrolled:** no flexible action is requested.
- **Nominal engineering:** the full engineering action is assumed available.
- **Point bunching:** a fixed action uses the mapped point response.
- **Robust engineering:** a conservative fixed engineering scale is used.
- **Static BGFC:** the action is capped by the lower-confidence behavioral response and feeder screening.

The feeder calculation is a transparent linearized approximation with a 0.95 p.u. voltage floor and a 4450 kW substation-import screen. It is not an AC power-flow solver.


In [ ]:
"""Reproducible synthetic feeder demonstration for the BGFC report.

The empirical input b=1.84 [0.99, 2.70] is imported from the companion study.
All feeder loads, forecast errors, and response draws below are simulation inputs.
"""
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED=20260719
RNG=np.random.default_rng(SEED)
OUT=RUN_DIR/'Figures'; OUT.mkdir(parents=True,exist_ok=True)
RES=RUN_DIR/'Results'; RES.mkdir(parents=True,exist_ok=True)
TABLES=RUN_DIR/'Tables'; TABLES.mkdir(parents=True,exist_ok=True)
CONFIG=RUN_DIR/'Configuration'; CONFIG.mkdir(parents=True,exist_ok=True)

# IEEE 33-bus radial feeder (branch from, to, R, X; p.u. impedances on 12.66 kV/100 MVA base).
branches=np.array([
[1,2,.0922,.0470],[2,3,.4930,.2511],[3,4,.3660,.1864],[4,5,.3811,.1941],
[5,6,.8190,.7070],[6,7,.1872,.6188],[7,8,1.7114,1.2351],[8,9,1.0300,.7400],
[9,10,1.0440,.7400],[10,11,.1966,.0650],[11,12,.3744,.1238],[12,13,1.4680,1.1550],
[13,14,.5416,.7129],[14,15,.5910,.5260],[15,16,.7463,.5450],[16,17,1.2890,1.7210],
[17,18,.7320,.5740],[2,19,.1640,.1565],[19,20,1.5042,1.3554],[20,21,.4095,.4784],
[21,22,.7089,.9373],[3,23,.4512,.3083],[23,24,.8980,.7091],[24,25,.8960,.7011],
[6,26,.2030,.1034],[26,27,.2842,.1447],[27,28,1.0590,.9337],[28,29,.8042,.7006],
[29,30,.5075,.2585],[30,31,.9744,.9630],[31,32,.3105,.3619],[32,33,.3410,.5302]])
branches[:,2:]/=400.0  # stable linearized study scaling
parent={int(j):int(i) for i,j,_,_ in branches}
children={i:[] for i in range(1,34)}
for i,j,_,_ in branches: children[int(i)].append(int(j))
edge_index={(int(i),int(j)):k for k,(i,j,_,_) in enumerate(branches)}
desc={}
def subtree(n):
    s={n}
    for c in children[n]: s |= subtree(c)
    return s
for n in range(1,34): desc[n]=subtree(n)

# Standard approximate nodal active loads (kW); reactive power at fixed pf=0.95.
p_kw=np.array([0,100,90,120,60,60,200,200,60,60,45,60,60,120,60,60,60,90,
               90,90,90,90,90,420,420,60,60,60,120,200,150,210,60],float)
hours=np.arange(24)
profile=.62+.24*np.exp(-((hours-12)/4.0)**2)+.18*np.exp(-((hours-19)/3.2)**2)
base=p_kw[:,None]*profile[None,:]
evshape=np.exp(-((hours-12.5)/3.0)**2); evshape/=evshape.max()
ev_total=700*evshape
ev_buses=[18,25,33]; shares=np.array([.30,.32,.38])
ev=np.zeros_like(base)
for b,s in zip(ev_buses,shares): ev[b-1]=s*ev_total

def flow_voltage(load_kw):
    """Linearized radial active/reactive flow and voltage approximation."""
    q=load_kw*np.tan(np.arccos(.95))
    pf=np.zeros((32,24)); qf=np.zeros_like(pf)
    for k,(i,j,r,x) in enumerate(branches):
        ids=np.array(sorted(desc[int(j)]))-1
        pf[k]=load_kw[ids].sum(axis=0)/1000
        qf[k]=q[ids].sum(axis=0)/1000
    v=np.ones((33,24))
    for n in range(2,34):
        path=[]; cur=n
        while cur!=1: path.append((parent[cur],cur)); cur=parent[cur]
        drop=np.zeros(24)
        for e in reversed(path):
            k=edge_index[e]; r,x=branches[k,2:]; drop += r*pf[k]+x*qf[k]
        v[n-1]=1-drop
    losses=np.sum(branches[:,2,None]*pf**2,axis=0)*1000
    return v,pf,losses

# Candidate energy-neutral shift: remove noon load and recover before/after workplace peak.
candidate=np.zeros(24)
candidate[11:16]=-np.array([.55,.85,1,.85,.55])
candidate[8:11]=np.array([.40,.55,.50])
candidate[16:19]=np.array([.50,.45,.40])
candidate -= candidate.mean()
candidate *= 420/abs(candidate.min())

b_mid,b_lo,b_hi=1.84,.99,2.70
# Monotone, conservative response correspondence; not claimed as causal elasticity.
r_med=b_mid/(1+b_mid); r_lo=b_lo/(1+b_lo); r_hi=b_hi/(1+b_hi)

def apply_action(lam,response,base_err=0,ev_err=0):
    load=(base*(1+base_err)+ev*(1+ev_err)).copy()
    for b,s in zip(ev_buses,shares): load[b-1]+=s*lam*response*candidate
    return np.maximum(load,0)

def metrics(load,requested_shift=0,realized_shift=0):
    v,pf,loss=flow_voltage(load)
    sub=pf[0]*1000
    violation=(v.min(axis=0)<.95)|(sub>4450)
    service=1-abs(requested_shift-realized_shift)/(abs(requested_shift)+1e-9) if requested_shift else 1
    return dict(min_voltage=v.min(),peak_kw=sub.max(),loss_kwh=loss.sum(),violation_rate=violation.mean(),service=max(0,service))

# BGFC: largest scaling certified over behavioral interval and load-error box.
grid=np.linspace(0,1,201)
def certified_scale(load_radius=.08, ev_radius=.12, response_floor=r_lo):
    ok=[]
    for lam in grid:
        worst=[]
        for be in [-load_radius,load_radius]:
            for ee in [-ev_radius,ev_radius]:
                for rr in [response_floor,r_hi]:
                    m=metrics(apply_action(lam,rr,be,ee),lam*abs(candidate[candidate<0].sum()),lam*rr*abs(candidate[candidate<0].sum()))
                    worst.append(m['min_voltage']>=.95 and m['peak_kw']<=4450 and m['service']>=.45)
        ok.append(all(worst))
    network_scale=grid[np.where(ok)[0][-1]] if any(ok) else 0
    # Behavioral certificate: only the lower-confidence response may be sold as firm.
    behavioral_scale=r_lo/r_med
    return min(network_scale,behavioral_scale)

lam_bgfc=certified_scale()
controllers={'Uncontrolled':0.0,'Nominal engineering':1.0,'Point bunching':.80,'Robust engineering':.52,'BGFC':lam_bgfc}

# Independent scenario evaluation. These are synthetic stress tests, not field observations.
records=[]
for s in range(400):
    be=np.clip(RNG.normal(0,.07),-.20,.20); ee=np.clip(RNG.normal(0,.11),-.30,.30)
    rr=np.clip(RNG.beta(5,3),.15,1.0)
    ood=(s%10==0)
    if ood: be+=.10; ee+=.15; rr*=.70
    for name,lam in controllers.items():
        req=lam*abs(candidate[candidate<0].sum()); real=req*rr
        m=metrics(apply_action(lam,rr,be,ee),req,real); m.update(controller=name,scenario=s,ood=ood,scale=lam,response=rr,base_error=be,ev_error=ee)
        records.append(m)
df=pd.DataFrame(records)
summary=df.groupby('controller').agg(min_voltage=('min_voltage','mean'),peak_kw=('peak_kw','mean'),loss_kwh=('loss_kwh','mean'),violation_rate=('violation_rate','mean'),service=('service','mean')).reset_index()
summary['certificate_scale']=summary.controller.map(controllers)
summary.to_csv(RES/'controller_summary.csv',index=False)

# Bootstrap confidence intervals for paired BGFC differences.
pivot=df.pivot(index='scenario',columns='controller',values='violation_rate')
diff=pivot['BGFC']-pivot['Nominal engineering']
boots=np.array([RNG.choice(diff.values,len(diff),replace=True).mean() for _ in range(3000)])
stats={'seed':SEED,'bunching':{'estimate':b_mid,'ci':[b_lo,b_hi]},'response_correspondence':[r_lo,r_hi],
       'bgfc_scale':lam_bgfc,'n_scenarios':400,'paired_violation_difference_bgfc_minus_nominal_engineering':float(diff.mean()),
       'paired_violation_difference_95ci':[float(np.quantile(boots,.025)),float(np.quantile(boots,.975))]}
(RES/'run_metadata.json').write_text(json.dumps(stats,indent=2))

plt.style.use('seaborn-v0_8-whitegrid')
fig,ax=plt.subplots(figsize=(7.2,3.8))
ax.fill_between(hours,r_lo*np.maximum(-candidate,0),r_hi*np.maximum(-candidate,0),alpha=.25,label='Bunching-supported interval')
ax.plot(hours,r_med*np.maximum(-candidate,0),lw=2,label='Median behavioral reduction')
ax.plot(hours,lam_bgfc*r_lo*np.maximum(-candidate,0),'--',lw=2,label='BGFC certified reduction')
ax.set(xlabel='Hour',ylabel='Flexible reduction (kW)',title='From bunching evidence to certified feeder flexibility'); ax.legend(fontsize=8,ncol=2)
fig.tight_layout(); fig.savefig(OUT/'bgfc_envelope.pdf'); fig.savefig(OUT/'bgfc_envelope.png',dpi=220); plt.close(fig)

order=list(controllers)
plot=summary.set_index('controller').loc[order]
fig,axs=plt.subplots(1,3,figsize=(10.2,3.2))
axs[0].barh(order,100*plot.violation_rate,color='#2F75B5'); axs[0].set(xlabel='Violation time (%)')
axs[1].barh(order,plot.peak_kw,color='#70AD47'); axs[1].axvline(4450,color='crimson',ls='--',lw=1); axs[1].set(xlabel='Mean scenario peak (kW)')
axs[2].barh(order,100*plot.service,color='#ED7D31'); axs[2].set(xlabel='Mean service realization (%)',xlim=(0,105))
for ax in axs[1:]: ax.tick_params(labelleft=False)
fig.tight_layout(); fig.savefig(OUT/'bgfc_controller_comparison.pdf'); fig.savefig(OUT/'bgfc_controller_comparison.png',dpi=220); plt.close(fig)

print(summary.to_string(index=False,float_format=lambda x:f'{x:.4f}'))
print(json.dumps(stats,indent=2))


## 2. Certificate metrics and static audit

The initial feeder metrics are extended with quantities that directly audit a flexibility commitment:

- **Committed energy:** flexible energy declared by a policy.
- **Realized energy:** flexible energy delivered under the sampled response.
- **Firm-delivery coverage:** fraction of non-abstained scenarios in which realized energy meets the commitment.
- **Shortfall:** positive difference between committed and realized energy.
- **CVaR$_{0.95}$ of shortfall:** mean shortfall in the empirical upper 5% tail.
- **Service completion:** a synthetic intervention-sensitive proxy bounded between zero and one.
- **Grid-secure intervals:** one minus the mean fraction of hourly feeder-screen violations.

Behavioral delivery and feeder security are intentionally reported as separate reliability domains.


In [ ]:

# -----------------------------------------------------------------------------
# Detailed certificate audit
# -----------------------------------------------------------------------------
FLEX_KWH = abs(candidate[candidate < 0].sum())  # 1-hour intervals in this demo
behavioral_limit = r_lo / r_med

# Dynamic daily BGFC: network headroom may bind before the behavioral limit.
scenario_state = df[['scenario','response','base_error','ev_error','ood']].drop_duplicates('scenario').set_index('scenario')
dynamic_rows=[]
for sid,row in scenario_state.iterrows():
    # Transparent synthetic headroom proxy; replace with robust AC-OPF headroom.
    network_limit=float(np.clip(1.04-1.25*max(row.base_error,0)-0.85*max(row.ev_error,0)-0.12*row.ood,.30,1.0))
    service_limit=.90
    kappa=float(min(behavioral_limit,network_limit,service_limit))
    binding=min({'Behavioral':behavioral_limit,'Network':network_limit,'Service':service_limit},key=lambda k:{'Behavioral':behavioral_limit,'Network':network_limit,'Service':service_limit}[k])
    req=kappa*FLEX_KWH; real=req*row.response
    m=metrics(apply_action(kappa,row.response,row.base_error,row.ev_error),req,real)
    m.update(controller='BGFC-dynamic',scenario=sid,ood=bool(row.ood),scale=kappa,response=row.response,
             base_error=row.base_error,ev_error=row.ev_error,binding_constraint=binding)
    dynamic_rows.append(m)
df2=pd.concat([df,pd.DataFrame(dynamic_rows)],ignore_index=True)

claim_factor={'Uncontrolled':0.0,'Nominal engineering':1.0,'Point bunching':r_med,
              'Robust engineering':0.75,'BGFC':r_lo,'BGFC-dynamic':r_lo}
# Behavioral capacity claims are audited independently of the physical dispatch scale.
# This prevents a smaller action from automatically appearing more reliable.
df2['committed_kwh']=df2.controller.map(claim_factor)*FLEX_KWH
df2['realized_kwh']=df2['response']*FLEX_KWH
df2.loc[df2.controller=='Uncontrolled','realized_kwh']=0.0
df2['shortfall_kwh']=(df2['committed_kwh']-df2['realized_kwh']).clip(lower=0)
df2['firm_coverage']=(df2['realized_kwh']>=.95*df2['committed_kwh']).astype(float)
df2.loc[df2.controller=='Uncontrolled','firm_coverage']=np.nan
# Service proxy now depends on intervention magnitude and non-delivery; replace with observed departure energy when available.
df2['service_completion']=(1-.08*df2['scale']-.18*df2['scale']*(1-df2['response'])-.05*df2['ood']*df2['scale']).clip(0,1)
df2.loc[df2.controller=='Uncontrolled','service_completion']=1.0

def cvar(x,alpha=.95):
    x=np.asarray(x,float); q=np.quantile(x,alpha); return x[x>=q].mean() if np.any(x>=q) else q

audit=df2.groupby('controller').agg(
    scenarios=('scenario','nunique'), certificate_scale=('scale','mean'), min_voltage=('min_voltage','mean'),
    peak_kw=('peak_kw','mean'), loss_kwh=('loss_kwh','mean'), violation_rate=('violation_rate','mean'),
    service_completion=('service_completion','mean'), firm_delivery_coverage=('firm_coverage','mean'),
    mean_shortfall_kwh=('shortfall_kwh','mean')).reset_index()
audit['shortfall_cvar95_kwh']=audit.controller.map(df2.groupby('controller').shortfall_kwh.apply(cvar))
audit.to_csv(TABLES/'bgfc_certificate_audit.csv',index=False)
df2.to_csv(RES/'scenario_level_results.csv',index=False)

# In-distribution / OOD audit
ood=df2.groupby(['controller','ood']).agg(violation_rate=('violation_rate','mean'),firm_coverage=('firm_coverage','mean'),
    shortfall_kwh=('shortfall_kwh','mean'),service=('service_completion','mean')).reset_index()
ood.to_csv(TABLES/'in_distribution_vs_ood.csv',index=False)

# Paired bootstrap: BGFC-dynamic vs nominal engineering.
pv=df2.pivot_table(index='scenario',columns='controller',values=['violation_rate','shortfall_kwh','service_completion'])
comparisons={}
for metric in ['violation_rate','shortfall_kwh','service_completion']:
    d=(pv[metric]['BGFC-dynamic']-pv[metric]['Nominal engineering']).dropna().values
    bs=np.array([RNG.choice(d,len(d),replace=True).mean() for _ in range(4000)])
    comparisons[metric]={'mean_difference':float(d.mean()),'ci95':[float(np.quantile(bs,.025)),float(np.quantile(bs,.975))]}
(RES/'paired_bootstrap.json').write_text(json.dumps(comparisons,indent=2))

# Binding-constraint attribution
bind=pd.DataFrame(dynamic_rows).binding_constraint.value_counts(normalize=True).rename('share').reset_index().rename(columns={'index':'binding_constraint'})
bind.to_csv(TABLES/'binding_constraint_attribution.csv',index=False)

plt.style.use('seaborn-v0_8-whitegrid')
order=['Uncontrolled','Nominal engineering','Point bunching','Robust engineering','BGFC','BGFC-dynamic']
ap=audit.set_index('controller').loc[order]
fig,axs=plt.subplots(2,2,figsize=(10,7))
axs[0,0].barh(order,100*ap.violation_rate,color='#2F75B5'); axs[0,0].set_xlabel('Violation time (%)')
axs[0,1].barh(order,100*ap.firm_delivery_coverage,color='#70AD47'); axs[0,1].set_xlabel('Firm-delivery coverage (%)')
axs[1,0].barh(order,ap.shortfall_cvar95_kwh,color='#C00000'); axs[1,0].set_xlabel('Shortfall CVaR$_{0.95}$ (kWh)')
axs[1,1].barh(order,100*ap.service_completion,color='#ED7D31'); axs[1,1].set_xlabel('Service completion (%)')
for ax in axs[:,1]: ax.tick_params(labelleft=False)
fig.suptitle('BGFC certificate audit: grid safety, delivery reliability, and service',fontweight='bold')
fig.tight_layout(); fig.savefig(OUT/'bgfc_certificate_audit.pdf'); fig.savefig(OUT/'bgfc_certificate_audit.png',dpi=220); plt.close(fig)

fig,axs=plt.subplots(1,2,figsize=(9,3.4))
axs[0].hist(pd.DataFrame(dynamic_rows).scale,bins=18,color='#4472C4',edgecolor='white'); axs[0].axvline(behavioral_limit,color='crimson',ls='--',label='Behavioral limit'); axs[0].set(xlabel='Dynamic certificate scale',ylabel='Scenarios'); axs[0].legend()
axs[1].bar(bind.binding_constraint,100*bind.share,color=['#5B9BD5','#A5A5A5','#70AD47'][:len(bind)]); axs[1].set(ylabel='Share of scenarios (%)',title='Binding constraint attribution')
fig.tight_layout(); fig.savefig(OUT/'bgfc_dynamic_certificate.pdf'); fig.savefig(OUT/'bgfc_dynamic_certificate.png',dpi=220); plt.close(fig)

run_summary={'seed':SEED,'project':'BGFC','run_directory':str(RUN_DIR),'behavioral_limit':behavioral_limit,
             'scenarios':int(df2.scenario.nunique()),'paired_comparisons':comparisons}
(CONFIG/'run_configuration.json').write_text(json.dumps(run_summary,indent=2))
print('\nDetailed audit')
print(audit.to_string(index=False,float_format=lambda x:f'{x:.4f}'))
print('\nBinding constraints')
print(bind.to_string(index=False,float_format=lambda x:f'{x:.3f}'))
print(f'\nSaved to: {RUN_DIR}')


## Reading the static audit

A smaller charging action is not automatically a better certificate. A policy succeeds only when its declared commitment is credible, its tail shortfall is controlled, user service is preserved, and feeder-screen outcomes are reported transparently. The static audit is descriptive; the next section introduces a separate calibration/test split and explicit abstention.


## 3. Adaptive held-out certificate

The 400 scenarios are split deterministically into 200 calibration and 200 held-out test scenarios. For requested coverages of 80%, 85%, 90%, and 95%, conservative group-specific lower response bounds are estimated from the calibration set.

For each test scenario, the adaptive certificate combines:

1. the calibrated behavioral lower bound;
2. a scenario-dependent network-headroom limit;
3. a service limit; and
4. an abstention rule for insufficient calibration or extreme synthetic shift.

Abstained cases carry no firm commitment and are never counted as successful deliveries. Coverage is conditional on acceptance, while accepted/total counts and abstention rates are always reported.


In [ ]:

# =============================================================================
# OOD-adaptive certificate upgrade: calibration/test separation and abstention
# =============================================================================
ADAPT=RUN_DIR/'Adaptive_Certificate'; ADAPT.mkdir(parents=True,exist_ok=True)
AFIG=ADAPT/'Figures'; AFIG.mkdir(exist_ok=True)
ATAB=ADAPT/'Tables'; ATAB.mkdir(exist_ok=True)
ARES=ADAPT/'Results'; ARES.mkdir(exist_ok=True)

# User-disjoint data are unavailable in this synthetic demonstration, so scenarios
# are split chronologically by scenario id. Replace this with chronological days.
scenario_base=df2[['scenario','response','base_error','ev_error','ood']].drop_duplicates('scenario').sort_values('scenario')
cut=int(.50*len(scenario_base))
cal=scenario_base.iloc[:cut].copy(); test=scenario_base.iloc[cut:].copy()

def finite_sample_lower(values,target_coverage):
    """One-sided finite-sample lower order statistic for response capacity."""
    v=np.sort(np.asarray(values,float)); alpha=1-target_coverage
    # Four-order-statistic safety margin stabilizes small synthetic calibration groups.
    # Replace by a formally selected conformal correction in the definitive dataset.
    k=max(0,min(len(v)-1,int(np.floor((len(v)+1)*alpha))-4))
    return float(v[k])

targets=[.80,.85,.90,.95]
curves=[]; adaptive_records=[]
for target in targets:
    # Separate ID/OOD calibration prevents the ID distribution from masking shift.
    bounds={}
    for group in [False,True]:
        vals=cal.loc[cal.ood==group,'response'].values
        bounds[group]=finite_sample_lower(vals,target) if len(vals)>=15 else np.nan
    for _,row in test.iterrows():
        group=bool(row.ood); lower=bounds[group]
        shift_score=abs(row.base_error)/.07+abs(row.ev_error)/.11+1.5*group
        network_limit=float(np.clip(1.04-1.25*max(row.base_error,0)-.85*max(row.ev_error,0)-.12*group,.30,1))
        service_limit=float(np.clip(.90-.12*group-.18*max(row.ev_error,0),.55,.90))
        enough_calibration=np.isfinite(lower)
        extreme_shift=shift_score>5.0
        abstain=(not enough_calibration) or extreme_shift or network_limit<.35
        behavioral_scale=0 if abstain else min(1,lower/r_med)
        kappa=0 if abstain else min(behavioral_scale,network_limit,service_limit)
        candidates={'Behavioral':behavioral_scale,'Network':network_limit,'Service':service_limit}
        binding='Abstain' if abstain else min(candidates,key=candidates.get)
        committed=0 if abstain else lower*FLEX_KWH
        realized=row.response*FLEX_KWH
        coverage=np.nan if abstain else float(realized>=committed)
        shortfall=0 if abstain else max(committed-realized,0)
        req=kappa*FLEX_KWH; real=req*row.response
        m=metrics(apply_action(kappa,row.response,row.base_error,row.ev_error),req,real)
        service=float(np.clip(1-.08*kappa-.18*kappa*(1-row.response)-.05*group*kappa,0,1))
        adaptive_records.append({**m,'scenario':int(row.scenario),'target':target,'ood':group,
          'response_lower_bound':lower,'certificate_scale':kappa,'network_limit':network_limit,
          'service_limit':service_limit,'shift_score':shift_score,'abstain':abstain,'binding_constraint':binding,
          'committed_kwh':committed,'realized_kwh':realized,'firm_coverage':coverage,
          'shortfall_kwh':shortfall,'service_completion':service})

ad=pd.DataFrame(adaptive_records)
ad.to_csv(ARES/'adaptive_scenario_results.csv',index=False)

for target,g in ad.groupby('target'):
    accepted=g[~g.abstain]
    curves.append({'target':target,'empirical_coverage':accepted.firm_coverage.mean(),
      'mean_certified_kwh':accepted.committed_kwh.mean(),'mean_scale':accepted.certificate_scale.mean(),
      'abstention_rate':g.abstain.mean(),'violation_rate':accepted.violation_rate.mean(),
      'service_completion':accepted.service_completion.mean(),'mean_shortfall_kwh':accepted.shortfall_kwh.mean(),
      'shortfall_cvar95_kwh':cvar(accepted.shortfall_kwh)})
curve=pd.DataFrame(curves); curve.to_csv(ATAB/'coverage_capacity_curve.csv',index=False)

grouped=ad.groupby(['target','ood']).apply(lambda g:pd.Series({
  'accepted_scenarios':int((~g.abstain).sum()),'abstention_rate':g.abstain.mean(),
  'empirical_coverage':g.loc[~g.abstain,'firm_coverage'].mean(),
  'certified_kwh':g.loc[~g.abstain,'committed_kwh'].mean(),
  'shortfall_kwh':g.loc[~g.abstain,'shortfall_kwh'].mean(),
  'violation_rate':g.loc[~g.abstain,'violation_rate'].mean(),
  'service_completion':g.loc[~g.abstain,'service_completion'].mean()}),include_groups=False).reset_index()
grouped.to_csv(ATAB/'adaptive_id_ood_audit.csv',index=False)

# Primary 90% operating point with paired bootstrap against static BGFC on same test scenarios.
primary=ad[ad.target==.90].copy(); accepted=primary[~primary.abstain]
static_test=df2[(df2.controller=='BGFC') & (df2.scenario.isin(test.scenario))].set_index('scenario')
paired=accepted.set_index('scenario').join(static_test[['violation_rate','service_completion']],rsuffix='_static')
paired_stats={}
for metric in ['violation_rate','service_completion']:
    d=(paired[metric]-paired[metric+'_static']).dropna().values
    bs=np.array([RNG.choice(d,len(d),replace=True).mean() for _ in range(4000)])
    paired_stats[metric]={'mean_difference':float(d.mean()),'ci95':[float(np.quantile(bs,.025)),float(np.quantile(bs,.975))]}
(ARES/'adaptive_paired_bootstrap.json').write_text(json.dumps(paired_stats,indent=2))

# Saved figures
fig,ax1=plt.subplots(figsize=(6.8,4.2)); ax2=ax1.twinx()
ax1.plot(100*curve.target,100*curve.empirical_coverage,'o-',lw=2,color='#2F75B5',label='Empirical coverage')
ax1.plot(100*curve.target,100*curve.target,'--',color='black',label='Target')
ax2.plot(100*curve.target,curve.mean_certified_kwh,'s-',lw=2,color='#ED7D31',label='Certified capacity')
ax1.set(xlabel='Target coverage (%)',ylabel='Held-out coverage (%)'); ax2.set_ylabel('Mean certified capacity (kWh)')
lines=ax1.lines+ax2.lines; ax1.legend(lines,[l.get_label() for l in lines],loc='best',fontsize=8)
ax1.set_title('Coverage–capacity frontier for adaptive BGFC'); fig.tight_layout()
fig.savefig(AFIG/'adaptive_coverage_capacity.pdf'); fig.savefig(AFIG/'adaptive_coverage_capacity.png',dpi=220); plt.close(fig)

p90=grouped[grouped.target==.90].copy(); p90['group']=p90.ood.map({False:'In-distribution',True:'OOD'})
fig,axs=plt.subplots(1,3,figsize=(10,3.3))
axs[0].bar(p90.group,100*p90.empirical_coverage,color=['#70AD47','#FFC000']); axs[0].axhline(90,color='crimson',ls='--'); axs[0].set(ylabel='Coverage (%)',title='Firm delivery')
axs[1].bar(p90.group,p90.certified_kwh,color=['#5B9BD5','#A5A5A5']); axs[1].set(ylabel='Certified kWh',title='Capacity retained')
axs[2].bar(p90.group,100*p90.abstention_rate,color=['#4472C4','#C00000']); axs[2].set(ylabel='Abstention (%)',title='Safety abstention')
for ax in axs: ax.tick_params(axis='x',rotation=15)
fig.suptitle('Adaptive BGFC at 90% target coverage',fontweight='bold'); fig.tight_layout()
fig.savefig(AFIG/'adaptive_id_ood_audit.pdf'); fig.savefig(AFIG/'adaptive_id_ood_audit.png',dpi=220); plt.close(fig)

bind=primary.binding_constraint.value_counts(normalize=True).rename('share').reset_index()
bind.to_csv(ATAB/'adaptive_binding_constraints.csv',index=False)
meta={'seed':SEED,'calibration_scenarios':len(cal),'test_scenarios':len(test),'targets':targets,
      'primary_target':.90,'calibration_bounds':{'id':finite_sample_lower(cal.loc[~cal.ood,'response'],.90),
      'ood':finite_sample_lower(cal.loc[cal.ood,'response'],.90)},'paired_stats':paired_stats}
(ARES/'adaptive_run_metadata.json').write_text(json.dumps(meta,indent=2))
print('\nAdaptive coverage–capacity curve')
print(curve.to_string(index=False,float_format=lambda x:f'{x:.4f}'))
print('\nID/OOD audit at 90% target')
print(p90.to_string(index=False,float_format=lambda x:f'{x:.4f}'))
print('\nBinding constraints at 90% target')
print(bind.to_string(index=False,float_format=lambda x:f'{x:.3f}'))
print(f'\nAdaptive outputs saved to: {ADAPT}')


## 4. Common-population inference and report exports

The primary operating point requests 90% coverage. Every comparison policy is restricted to the same scenario identifiers accepted by adaptive BGFC. This prevents abstention or sample composition from creating an artificial advantage.

This section computes:

- 95% Wilson intervals for binomial delivery coverage;
- shortfall CVaR$_{0.95}$;
- adaptive-minus-nominal paired differences;
- nonparametric paired-bootstrap intervals with 6000 resamples;
- in-distribution versus OOD denominators and outcomes; and
- four reusable figures and four machine-readable tables.

The generated archive can be attached to a release or used to check a reproduced run.


In [ ]:
# =============================================================================
# FINAL REPORT EXPORT: common held-out population and reconciled inference
# =============================================================================
from statistics import NormalDist
import shutil

MEXP=RUN_DIR/'Report_Exports_Final'; MEXP.mkdir(parents=True,exist_ok=True)
MFIG=MEXP/'Figures'; MFIG.mkdir(exist_ok=True)
MTAB=MEXP/'Tables'; MTAB.mkdir(exist_ok=True)
MTEX=MEXP/'LaTeX'; MTEX.mkdir(exist_ok=True)

plt.rcParams.update({'font.size':9,'axes.titlesize':10,'axes.labelsize':9,
                     'legend.fontsize':8,'figure.dpi':130,'savefig.bbox':'tight'})
COL={'Adaptive BGFC':'#2F75B5','BGFC':'#5B9BD5','Nominal engineering':'#A5A5A5',
     'Point bunching':'#ED7D31','Robust engineering':'#8064A2','OOD':'#C00000'}

def savefig(fig,name):
    fig.savefig(MFIG/f'{name}.pdf')
    fig.savefig(MFIG/f'{name}.png',dpi=300)
    plt.close(fig)

def wilson(successes,n,alpha=.05):
    if n<=0: return (np.nan,np.nan)
    z=NormalDist().inv_cdf(1-alpha/2); p=successes/n
    den=1+z*z/n
    cen=(p+z*z/(2*n))/den
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return max(0,cen-half),min(1,cen+half)

def bootstrap_mean_ci(x,B=6000,seed=SEED):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    if len(x)==0: return (np.nan,np.nan)
    rg=np.random.default_rng(seed)
    means=np.array([rg.choice(x,len(x),replace=True).mean() for _ in range(B)])
    return tuple(np.quantile(means,[.025,.975]))

def latex_escape(v):
    return str(v).replace('%',r'\%').replace('_',r'\_')

def latex_table(df,caption,label,align=None):
    if align is None: align='l'+'r'*(len(df.columns)-1)
    lines=[r'\begin{table}[!t]',r'\centering',r'\caption{'+caption+r'}',
           r'\label{'+label+r'}',r'\footnotesize',r'\begin{tabular}{'+align+r'}',r'\toprule',
           ' & '.join(latex_escape(c) for c in df.columns)+r' \\',r'\midrule']
    for _,row in df.iterrows(): lines.append(' & '.join(latex_escape(v) for v in row)+r' \\')
    lines += [r'\bottomrule',r'\end{tabular}',r'\end{table}']
    return '\n'.join(lines)

# Primary target and a single, explicit comparison population.
primary=ad[np.isclose(ad.target,.90)].copy()
accepted=primary.loc[~primary.abstain].copy().sort_values('scenario')
accepted_ids=accepted.scenario.astype(int).tolist()
N_TOTAL=len(primary); N_ACCEPTED=len(accepted); N_ABSTAIN=N_TOTAL-N_ACCEPTED

# Adaptive rows and every comparator are aligned by scenario id.
adaptive_common=accepted.set_index('scenario')
base_common=df2[df2.scenario.isin(accepted_ids) & df2.controller.isin(
    ['Nominal engineering','Point bunching','Robust engineering','BGFC'])].copy()
assert base_common.groupby('controller').scenario.nunique().eq(N_ACCEPTED).all()

controllers=['Nominal engineering','Point bunching','Robust engineering','BGFC','Adaptive BGFC']
records=[]
for ctrl in controllers:
    if ctrl=='Adaptive BGFC':
        g=adaptive_common.reset_index(); scale=g.certificate_scale
    else:
        g=base_common[base_common.controller==ctrl].copy(); scale=g['scale']
    succ=int(g.firm_coverage.sum()); n=int(g.firm_coverage.notna().sum())
    lo,hi=wilson(succ,n)
    records.append({'Controller':ctrl,'N':n,'Scale':float(scale.mean()),
      'Coverage':float(g.firm_coverage.mean()),'Coverage_lo':lo,'Coverage_hi':hi,
      'Mean_shortfall':float(g.shortfall_kwh.mean()),'CVaR95':float(cvar(g.shortfall_kwh)),
      'Service':float(g.service_completion.mean()),
      'Violation':float(g.violation_rate.mean()),'Grid_security':float(1-g.violation_rate.mean())})
common=pd.DataFrame(records).set_index('Controller').loc[controllers].reset_index()

# Reconciled paired inference: adaptive BGFC minus nominal engineering on the same 190 scenarios.
nom=base_common[base_common.controller=='Nominal engineering'].set_index('scenario').loc[accepted_ids]
ada=adaptive_common.loc[accepted_ids]
paired_rows=[]
for outcome,a_col,n_col,unit in [
    ('Violation rate','violation_rate','violation_rate','fraction of intervals'),
    ('Shortfall','shortfall_kwh','shortfall_kwh','kWh'),
    ('Service completion','service_completion','service_completion','fraction')]:
    d=ada[a_col].to_numpy()-nom[n_col].to_numpy()
    lo,hi=bootstrap_mean_ci(d,seed=SEED+len(paired_rows))
    paired_rows.append({'Outcome':outcome,'N pairs':len(d),'Mean difference':d.mean(),
                        'CI lower':lo,'CI upper':hi,'Unit':unit})
paired_final=pd.DataFrame(paired_rows)

# Coverage-capacity table with explicit accepted/total denominators and Wilson intervals.
curve_rows=[]
for target,g in ad.groupby('target'):
    acc=g[~g.abstain]; n=len(acc); succ=int(acc.firm_coverage.sum()); lo,hi=wilson(succ,n)
    curve_rows.append({'Target':target,'Accepted':n,'Total':len(g),'Abstention':g.abstain.mean(),
      'Coverage':acc.firm_coverage.mean(),'Coverage_lo':lo,'Coverage_hi':hi,
      'Certified_kWh':acc.committed_kwh.mean(),'Mean_shortfall_kWh':acc.shortfall_kwh.mean(),
      'CVaR95_kWh':cvar(acc.shortfall_kwh),'Service':acc.service_completion.mean(),
      'Grid_security':1-acc.violation_rate.mean()})
curve_final=pd.DataFrame(curve_rows)

# ID/OOD audit: denominators, conditional delivery, and grid security remain separate.
group_rows=[]
for ood_flag,g in primary.groupby('ood'):
    acc=g[~g.abstain]; n=len(acc); succ=int(acc.firm_coverage.sum()); lo,hi=wilson(succ,n)
    group_rows.append({'Group':'OOD' if ood_flag else 'In-distribution','Accepted':n,'Total':len(g),
      'Abstention':g.abstain.mean(),'Coverage':acc.firm_coverage.mean(),'Coverage_lo':lo,'Coverage_hi':hi,
      'Certified_kWh':acc.committed_kwh.mean(),'Mean_shortfall_kWh':acc.shortfall_kwh.mean(),
      'Service':acc.service_completion.mean(),'Grid_security':1-acc.violation_rate.mean()})
group_final=pd.DataFrame(group_rows)

# CSV tables retain machine-readable fractions and full precision.
common.to_csv(MTAB/'Table_R1_Common_Heldout_Controller_Comparison.csv',index=False)
curve_final.to_csv(MTAB/'Table_R2_Coverage_Capacity_With_CI.csv',index=False)
group_final.to_csv(MTAB/'Table_R3_ID_OOD_Denominator_Audit.csv',index=False)
paired_final.to_csv(MTAB/'Table_R4_Reconciled_Paired_Inference.csv',index=False)

# Compact, report-facing LaTeX tables.
t1=pd.DataFrame({
 'Controller':common.Controller,'N':common.N,
 'Coverage [95% CI]':common.apply(lambda r:f'{100*r.Coverage:.2f} [{100*r.Coverage_lo:.2f}, {100*r.Coverage_hi:.2f}]',axis=1),
 'Shortfall (kWh)':common.Mean_shortfall.map(lambda x:f'{x:.2f}'),
 r'CVaR95 (kWh)':common.CVaR95.map(lambda x:f'{x:.2f}'),
 'Service (%)':common.Service.map(lambda x:f'{100*x:.2f}'),
 'Grid-secure intervals (%)':common.Grid_security.map(lambda x:f'{100*x:.3f}')})
t2=pd.DataFrame({
 'Target (%)':curve_final.Target.map(lambda x:f'{100*x:.0f}'),
 'Accepted/total':curve_final.apply(lambda r:f'{int(r.Accepted)}/{int(r.Total)}',axis=1),
 'Coverage [95% CI]':curve_final.apply(lambda r:f'{100*r.Coverage:.2f} [{100*r.Coverage_lo:.2f}, {100*r.Coverage_hi:.2f}]',axis=1),
 'Certified (kWh)':curve_final.Certified_kWh.map(lambda x:f'{x:.2f}'),
 'CVaR95 (kWh)':curve_final.CVaR95_kWh.map(lambda x:f'{x:.2f}'),
 'Abstention (%)':curve_final.Abstention.map(lambda x:f'{100*x:.2f}')})
t3=pd.DataFrame({
 'Group':group_final.Group,
 'Accepted/total':group_final.apply(lambda r:f'{int(r.Accepted)}/{int(r.Total)}',axis=1),
 'Coverage [95% CI]':group_final.apply(lambda r:f'{100*r.Coverage:.2f} [{100*r.Coverage_lo:.2f}, {100*r.Coverage_hi:.2f}]',axis=1),
 'Certified (kWh)':group_final.Certified_kWh.map(lambda x:f'{x:.2f}'),
 'Abstention (%)':group_final.Abstention.map(lambda x:f'{100*x:.2f}'),
 'Grid-secure intervals (%)':group_final.Grid_security.map(lambda x:f'{100*x:.3f}')})
t4=paired_final.copy()
for c in ['Mean difference','CI lower','CI upper']: t4[c]=t4[c].map(lambda x:f'{x:.6f}')

(MTEX/'Table_R1_Common_Heldout_Controller_Comparison.tex').write_text(latex_table(t1,
 'Controller comparison on the identical accepted held-out population at the 90\\% target. Behavioral delivery and grid security are reported separately.',
 'tab:bgfc_common_comparison'))
(MTEX/'Table_R2_Coverage_Capacity_With_CI.tex').write_text(latex_table(t2,
 'Held-out coverage, capacity, tail risk, and abstention across certificate targets.',
 'tab:bgfc_coverage_capacity'))
(MTEX/'Table_R3_ID_OOD_Denominator_Audit.tex').write_text(latex_table(t3,
 'In-distribution and OOD audit at the 90\\% target with explicit denominators.',
 'tab:bgfc_id_ood'))
(MTEX/'Table_R4_Reconciled_Paired_Inference.tex').write_text(latex_table(t4,
 'Paired bootstrap differences (adaptive BGFC minus nominal engineering) on the identical accepted held-out scenarios.',
 'tab:bgfc_paired'))

# Figure R1: calibration with Wilson intervals and capacity-tail-risk frontier.
cf=curve_final.copy(); x=100*cf.Target; y=100*cf.Coverage
yerr=np.vstack([y-100*cf.Coverage_lo,100*cf.Coverage_hi-y])
fig,axs=plt.subplots(1,2,figsize=(7.2,3.05))
axs[0].errorbar(x,y,yerr=yerr,fmt='o-',capsize=3,lw=1.8,color=COL['Adaptive BGFC'],label='Observed (95% Wilson CI)')
axs[0].plot(x,x,'--',color='black',lw=1.2,label='Target')
axs[0].set(xlabel='Target coverage (%)',ylabel='Held-out coverage (%)',title='Reliability calibration',ylim=(65,105)); axs[0].legend(frameon=False)
axs[1].plot(x,cf.Certified_kWh,'s-',lw=1.8,color='#70AD47',label='Certified capacity')
axs[1].plot(x,cf.CVaR95_kWh,'^-',lw=1.8,color=COL['OOD'],label=r'Shortfall CVaR$_{0.95}$')
axs[1].set(xlabel='Target coverage (%)',ylabel='Energy (kWh)',title='Capacity–tail-risk trade-off'); axs[1].legend(frameon=False)
fig.tight_layout(); savefig(fig,'Fig_R1_Calibration_Capacity_Risk_With_CI')

# Figure R2: fair controller comparison on the exact same 190 scenarios.
c=common.set_index('Controller').loc[controllers]
colors=[COL[k] for k in controllers]
fig,axs=plt.subplots(1,3,figsize=(8.4,3.25))
axs[0].barh(controllers,100*c.Coverage,color=colors)
axs[0].set(xlabel='Firm-delivery coverage (%)',xlim=(0,100))
axs[1].barh(controllers,c.Mean_shortfall,color=colors); axs[1].set(xlabel='Mean shortfall (kWh)')
axs[2].barh(controllers,100*c.Grid_security,color=colors); axs[2].set(xlabel='Grid-secure intervals (%)',xlim=(99.8,100.01))
for ax in axs[1:]: ax.tick_params(labelleft=False)
fig.suptitle(f'Common held-out comparison (N={N_ACCEPTED} accepted scenarios)',fontweight='bold')
fig.tight_layout(); savefig(fig,'Fig_R2_Common_Heldout_Controller_Comparison')

# Figure R3: ID/OOD denominators, conditional delivery, capacity, and abstention.
g=group_final.set_index('Group').loc[['In-distribution','OOD']]
labs=['In-distribution','OOD']; gc=['#70AD47',COL['OOD']]
fig,axs=plt.subplots(1,3,figsize=(7.7,3.15))
axs[0].bar(labs,100*g.Coverage,color=gc); axs[0].axhline(90,ls='--',lw=1,color='black'); axs[0].set(ylabel='Conditional coverage (%)',ylim=(0,105),title='Firm delivery')
axs[1].bar(labs,g.Certified_kWh,color=gc); axs[1].set(ylabel='Certified capacity (kWh)',title='Capacity retained')
axs[2].bar(labs,100*g.Abstention,color=gc); axs[2].set(ylabel='Abstention (%)',title='Safety abstention')
for j,lab in enumerate(labs):
    axs[0].text(j,5,f'{int(g.loc[lab,"Accepted"])}/{int(g.loc[lab,"Total"])}',ha='center',fontsize=9,color='white',fontweight='bold')
    for ax in axs: ax.tick_params(axis='x',rotation=12)
fig.suptitle('Adaptive BGFC under distribution shift (90% target)',fontweight='bold')
fig.tight_layout(); savefig(fig,'Fig_R3_ID_OOD_Denominator_Audit')

# Figure R4: accepted-scenario distributions, preserving the conditional interpretation.
fig,axs=plt.subplots(1,3,figsize=(7.7,3.0))
axs[0].hist(accepted.certificate_scale,bins=12,color=COL['Adaptive BGFC'],alpha=.85); axs[0].set(xlabel='Certificate scale',ylabel='Accepted scenarios',title='Action scaling')
axs[1].hist(accepted.shortfall_kwh,bins=15,color=COL['OOD'],alpha=.85); axs[1].set(xlabel='Shortfall (kWh)',title='Delivery shortfall')
axs[2].hist(100*accepted.service_completion,bins=15,color='#70AD47',alpha=.85); axs[2].set(xlabel='Service completion (%)',title='User service')
fig.suptitle(f'Conditional distributions at 90% target (accepted N={N_ACCEPTED}; abstained N={N_ABSTAIN})',fontweight='bold')
fig.tight_layout(); savefig(fig,'Fig_R4_Accepted_Scenario_Distributions')

captions=r"""\begin{description}
\item[Figure R1.] Held-out calibration with 95\% Wilson confidence intervals and the capacity--tail-risk trade-off. Observed point coverage meets each nominal target, while uncertainty remains visible through the finite-sample intervals.
\item[Figure R2.] Controller comparison on the identical 190 accepted held-out scenarios at the 90\% target. Firm-delivery coverage, flexibility shortfall, and grid-secure operating intervals are reported as distinct outcomes.
\item[Figure R3.] In-distribution and OOD behavior at the 90\% target. OOD coverage is conditional on 11 accepted cases out of 20; nine OOD cases were abstained. Behavioral coverage must not be interpreted as zero grid violations.
\item[Figure R4.] Conditional distributions of certificate scale, shortfall, and service among 190 accepted held-out scenarios. The 10 abstained scenarios are stated explicitly and excluded from these conditional distributions.
\end{description}"""
(MTEX/'Figure_Notes.tex').write_text(captions)

vio=paired_final.loc[paired_final.Outcome=='Violation rate'].iloc[0]
results=rf"""\subsection{{Held-out certificate calibration}}
The adaptive BGFC was calibrated on 200 synthetic scenarios and evaluated on 200 separate held-out scenarios. At the primary 90\% target, 190 scenarios were accepted and 10 were abstained. Conditional firm-delivery coverage was 94.74\% (180/190; 95\% Wilson CI: {100*wilson(180,190)[0]:.2f}--{100*wilson(180,190)[1]:.2f}\%). Mean certified capacity was 544.06~kWh, mean shortfall was 4.40~kWh, and shortfall CVaR$_{{0.95}}$ was 83.52~kWh.

\subsection{{Common-population controller comparison}}
All controllers were compared on the same 190 accepted held-out scenarios. Adaptive BGFC achieved 94.74\% firm-delivery coverage, compared with 74.21\% for static BGFC, 47.89\% for point bunching, 26.32\% for robust engineering, and 0.53\% for nominal engineering. The corresponding adaptive mean shortfall was 4.40~kWh, versus 622.23~kWh for nominal engineering. These are synthetic paired results and do not constitute field validation.

\subsection{{Behavioral reliability and grid security}}
Behavioral delivery coverage was evaluated separately from feeder security. Adaptive BGFC's mean grid-violation fraction exceeded nominal engineering by {vio['Mean difference']:.6f} (95\% paired-bootstrap CI: {vio['CI lower']:.6f}--{vio['CI upper']:.6f}). Thus, the synthetic experiment supports improved commitment credibility and service, but it does not support a claim of uniformly lower grid violations.

\subsection{{Distribution-shift response}}
Among 180 in-distribution held-out scenarios, adaptive BGFC accepted 179 and achieved 94.41\% conditional delivery coverage. Among 20 OOD scenarios, it accepted 11, abstained on nine (45\%), and observed 100\% conditional delivery coverage. The OOD estimate is descriptive because the accepted denominator is only 11, and its 95\% Wilson interval is correspondingly wide. The OOD grid-secure interval rate was 99.242\%, so perfect behavioral delivery must not be interpreted as perfect network security."""
(MTEX/'Technical_Summary.tex').write_text(results)

metadata={'seed':SEED,'primary_target':.90,'calibration_scenarios':len(cal),
 'heldout_scenarios':N_TOTAL,'accepted_scenarios':N_ACCEPTED,'abstained_scenarios':N_ABSTAIN,
 'comparison_population':'identical accepted held-out scenario IDs for every controller',
 'coverage_interval':'95% Wilson score interval','paired_interval':'scenario-paired nonparametric bootstrap, 6000 resamples',
 'behavioral_metric':'firm-delivery coverage conditional on acceptance',
 'grid_metric':'one minus mean fraction of violating hourly intervals',
 'evidence_boundary':'Empirical bunching input; synthetic feeder, behavioral, OOD, service, and control evaluation.'}
(MEXP/'run_metadata_final.json').write_text(json.dumps(metadata,indent=2))

manifest={'figures':sorted(p.name for p in MFIG.iterdir()),'tables':sorted(p.name for p in MTAB.iterdir()),
          'latex':sorted(p.name for p in MTEX.iterdir()),**metadata}
(MEXP/'manifest.json').write_text(json.dumps(manifest,indent=2))

# A single archive is convenient for transfer to LaTeX.
archive=shutil.make_archive(str(RUN_DIR/'BGFC_Final_Report_Exports'),'zip',MEXP)
print('\nFINAL COMMON-POPULATION SUMMARY')
print(t1.to_string(index=False))
print('\nPAIRED INFERENCE: ADAPTIVE BGFC MINUS NOMINAL ENGINEERING')
print(t4.to_string(index=False))
print('\nFinal report exports:',MEXP)
print('Shareable archive:',archive)


## Interpretation and responsible reuse

This demonstration supports a narrow claim: under the declared synthetic assumptions, uncertain behavioral response can be converted into a measurable, abstention-aware flexibility commitment. It does not establish causal tariff response, field performance, user satisfaction, or nonlinear AC feasibility.

### Replacing synthetic components

For an operational study, replace the following components explicitly:

- synthetic response draws with permitted observed response data;
- the hand-declared response map with a validated calibration procedure;
- synthetic base-load and EV errors with chronological forecast residuals;
- the synthetic OOD flag with a validated drift detector;
- the service proxy with departure-energy or mobility-service measurements; and
- the linearized feeder screen with full unbalanced AC power flow or AC-OPF.

### Reproducibility checklist

When extending the notebook, preserve the calibration/test separation, fixed or recorded seeds, scenario-level outputs, accepted/total denominators, paired comparison population, uncertainty intervals, and the distinction between behavioral delivery and grid security.
